# Augsburg through its streets
## Day 02 · Lines · #30DayMapChallenge 2026

Augsburg's OpenStreetMap street network, displayed over the municipal
boundary with coral lines on a blue background.

The network includes roads, pedestrian routes, paths and trails.
Both layers are projected to EPSG:32632 for mapping and the scale bar.

**Map by Maria Victoria León Parra**

### Required local files

No local input files are required for the first run.
OpenStreetMap data is downloaded automatically.

The street network is saved in the project's `data/` directory:

- `augsburg_all_streets.graphml`

Subsequent runs load this local copy instead of downloading it again.

In [ ]:
%pip install osmnx matplotlib

In [ ]:
import osmnx as ox
import matplotlib.pyplot as plt

print("OSMnx:", ox.__version__)

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Working directory:", BASE_DIR)

In [ ]:
graph_file = DATA_DIR / "augsburg_all_streets.graphml"

if graph_file.exists():
    G = ox.load_graphml(graph_file)
    print("Data loaded from the local copy.")
else:
    G = ox.graph_from_place(
        "Augsburg, Bavaria, Germany",
        network_type="all"
    )
    ox.save_graphml(G, graph_file)
    print("Download completed and saved.")

print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

In [ ]:
# Download the municipal boundary and project it to meters
admin = ox.geocode_to_gdf("Augsburg, Bavaria, Germany")
admin = admin.to_crs(epsg=32632)

# Use the same coordinate system for the streets
edges = edges.to_crs(epsg=32632)

In [ ]:
# Convert the network into GeoDataFrames and project to meters
nodes, edges = ox.graph_to_gdfs(G)
edges = edges.to_crs(epsg=32632)
admin = admin.to_crs(epsg=32632)

# Define map colors
line_color = "#E0696E"
background = "#D1D1D1"

# Create the figure and reserve space for the footer
fig, ax = plt.subplots(figsize=(12, 12))
fig.patch.set_facecolor(background)
ax.set_facecolor(background)
fig.subplots_adjust(top=0.88, bottom=0.12)

# Draw the municipal boundary beneath the streets
admin.plot(
    ax=ax,
    facecolor="#14396A",
    edgecolor="none",
    zorder=1
)

# Draw the street network
edges.plot(
    ax=ax,
    color=line_color,
    linewidth=0.5,
    alpha=0.9,
    zorder=2
)

# Add the title
fig.suptitle("Day 2 - Lines", fontsize=30, color= "#14396A", y=0.92)

# Add the footer
fig.text(
    0.5, 0.12,
    "#30DayMapChallenge | 2026 | Day - 02 | Augsburg through its streets | "
    "Map by Maria Victoria León Parra",
    ha="center",
    fontsize=9,
    color="#14396A"
)

# Hide the axes
ax.set_axis_off()

# Add a north arrow
ax.annotate(
    "",
    xy=(0.91, 0.95),
    xytext=(0.91, 0.86),
    xycoords="axes fraction",
    arrowprops=dict(
        facecolor="#14396A",
        edgecolor="#14396A",
        width=4,
        headwidth=14,
        headlength=18
    ),
    zorder=5
)

ax.text(
    0.91, 0.97, "N",
    transform=ax.transAxes,
    ha="center",
    fontsize=14,
    fontweight="bold",
    color="#14396A"
)

# Preserve the map extent before adding the scale bar
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# Position a 2 km scale bar in projected map units (meters)
x = xmin + 0.04 * (xmax - xmin)
y = ymin + 0.05 * (ymax - ymin)
tick_height = 0.012 * (ymax - ymin)

ax.plot(
    [x, x + 2000], [y, y],
    color="#14396A",
    linewidth=2,
    zorder=5
)

for offset, label in [(0, "0"), (1000, "1"), (2000, "2 km")]:
    ax.plot(
        [x + offset, x + offset],
        [y, y + tick_height],
        color="#14396A",
        linewidth=2,
        zorder=5
    )
    ax.text(
        x + offset,
        y + 1.5 * tick_height,
        label,
        ha="center",
        fontsize=10,
        color="#14396A",
        zorder=5
    )

# Keep the original map extent
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

plt.show()

In [ ]:
# Save the map as a high-resolution PNG
output_file = OUTPUT_DIR / "day02_lines_augsburg.png"

fig.savefig(
    output_file,
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.3,
    facecolor=fig.get_facecolor()
)

print("Map saved to:", output_file)

## Sources and limitations

- Street network and municipal boundary: © OpenStreetMap contributors, ODbL.
  https://www.openstreetmap.org/copyright
- Workflow inspiration: Milan Janosov's Visualizing Road Networks tutorial.
  https://milanjanosov.substack.com/p/milans-data-stories-016-visualizing

OpenStreetMap coverage may be incomplete. The network is downloaded
using `network_type="all"`, including roads, pedestrian routes, paths
and trails selected by OSMnx. This does not imply access for all
transport modes.

Street segments crossing the municipal boundary may extend slightly
beyond it. All streets use the same color and do not represent
traffic levels, travel times or road classifications.

Both layers are projected to EPSG:32632. The scale bar uses meters,
and the north arrow indicates grid north.

Cached data and generated outputs should be excluded from Git.
Review source-data licences before redistributing files.